# Datenvalidierung

Prueft PDFs, Markdown-Konvertierung und Labels, bevor die Pipeline trainiert.
Alle Berichte werden unter `daten/validation/` gespeichert.

Voraussetzungen:
- `01_cockpit.py` im Projektordner (gemeinsame Konfiguration mit dem Pipeline-Notebook)
- `daten/labels/Repro_data_ges.csv`
- PDFs unter `daten/pdfs/<jahr>/`, fuer Abschnitt 3 zusaetzlich `daten/markdown/<jahr>/`

Das Notebook nutzt keine Variablen aus anderen Notebooks und laeuft mit "Restart & Run All".

In [ ]:
# Abhaengigkeiten aus der festgelegten Versionsliste installieren.
# requirements.txt einmalig aus der Arbeitsumgebung erzeugen: pip freeze > requirements.txt
%pip install -r requirements.txt

In [1]:
# Gemeinsame Konfiguration laden. -i fuehrt das Cockpit im Notebook-Namensraum aus,
# damit dessen Funktionen spaeter gesetzte globale Variablen sehen.
%run -i 01_cockpit.py

from IPython.display import display

VALIDATION_ROOT = BASE_DIR / "daten" / "validation"
VALIDATION_ROOT.mkdir(parents=True, exist_ok=True)

/home/Mike-bazz/.local/lib/python3.14/site-packages/langchain_core/utils/pydantic.py:41: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


Projektstruktur wird initialisiert
Quelle: pdf_raw | Embedding: mxbai-embed-large | LLM: llama3.2
Chunking: recursive_1200c_200o_norefs | Pooling: max | Seed: 42
Vorhersageziele: 9 Spalten

Daten-Audit
Label-Datei: IMWUT_Dataset_original.csv (SHA-256 ee5cf2727b11...)

ID-Nummerierung je Jahr
2023: 1..205, 205 Eintraege, lueckenlos
2024: 1..209, 208 Eintraege, bekannte Luecken [165]
2025: 1..243, 242 Eintraege, bekannte Luecken [173]
Ausgeschlossen: 2023_Dec_188 (Defektes Font-Encoding, Text nicht extrahierbar; aus Tabelle entfernt)

Zusammenfassung Datenbasis
Theoretisches Maximum (ID-Zaehler): 657
Annotiert:                          655
Verwendet:                          654
Vorhandene PDFs (verwendet):        654
Fehlende PDFs:                      0
PDFs ohne Label-Eintrag:            0

Label-Mapping
code_required: 4 Eintraege umkodiert
data_required: 6 Eintraege umkodiert
hardware_required: 60 Eintraege umkodiert
software_required: 68 Eintraege umkodiert
code_available: 84 Eintra

## 1. PDF-Integritaet

Prueft fuer jede verwendete Studie, ob die PDF existiert, sich oeffnen laesst,
nicht verschluesselt ist und Seiten enthaelt.

In [ ]:
MIN_PDF_BYTES = 1024  # Kleinere Dateien sind praktisch leer

fitz.TOOLS.mupdf_display_errors(False)


def check_pdf(pdf_path: Path) -> tuple[str, str]:
    """Liefert (Status, Grund) fuer eine PDF."""
    if not pdf_path.exists():
        return "fehlt", "Datei nicht vorhanden"
    if pdf_path.stat().st_size < MIN_PDF_BYTES:
        return "leer", f"kleiner als {MIN_PDF_BYTES} Byte"
    try:
        with fitz.open(pdf_path) as doc:
            if doc.needs_pass:
                return "verschluesselt", "Passwort benoetigt"
            if doc.page_count == 0:
                return "leer", "0 Seiten"
            return "ok", f"{doc.page_count} Seiten"
    except Exception as e:
        return "beschaedigt", str(e)


pdf_report = []
for row in tqdm(master_label_df[["paper_id", "year"]].itertuples(index=False),
                total=len(master_label_df), desc="PDF-Check"):
    status, grund = check_pdf(STUDIEN_ROOT / row.year / f"{row.paper_id}.pdf")
    pdf_report.append({"paper_id": row.paper_id, "year": row.year, "status": status, "grund": grund})

pdf_report_df = pd.DataFrame(pdf_report)
pdf_report_df.to_csv(VALIDATION_ROOT / "pdf_report.csv", index=False)

print(pdf_report_df["status"].value_counts().to_string())
auffaellig = pdf_report_df[pdf_report_df["status"] != "ok"]
if not auffaellig.empty:
    display(auffaellig)

## 2. Markdown-Qualitaet

Vollstaendigkeit der Konvertierung und Kennzahlen zur Extraktionsqualitaet.
U+FFFD (Ersetzungszeichen) entsteht, wenn Glyphen keinem Unicode-Zeichen
zugeordnet werden koennen, und dient als Indikator fuer defektes Font-Encoding.

Die Schwellen sind Setzungen. Vor der Verwendung in der Arbeit an der
ausgegebenen Verteilung pruefen und begruenden.

In [ ]:
MIN_CHARS = 500                 # Darunter: praktisch kein Text extrahiert
MAX_UFFFD_ABSOLUT = 150         # Ersetzungszeichen pro Dokument
MAX_UFFFD_ANTEIL = 0.02         # Anteil am Text (0.02 = 2 %)
MAX_BILDER_BEI_WENIG_TEXT = 8   # Viele Bild-Tags ...
MIN_WOERTER_BEI_BILDERN = 1500  # ... bei wenig Fliesstext

# pymupdf4llm ersetzt Bilder ohne write_images durch Platzhalter wie
# "**==> picture [307 x 24] intentionally omitted <==**"; Markdown-Bildlinks zaehlen ebenfalls.
IMAGE_TAG = re.compile(r"==> picture \[[^\]]*\] intentionally omitted <==|!\[.*?\]\(.*?\)")


def bewerte_markdown(text: str) -> dict:
    chars = len(text)
    woerter = len(text.split())
    n_ufffd = text.count("\ufffd")
    anteil = n_ufffd / chars if chars else 0.0
    bilder = len(IMAGE_TAG.findall(text))

    if chars < MIN_CHARS:
        status = "defekt: kaum Text"
    elif n_ufffd > MAX_UFFFD_ABSOLUT or anteil > MAX_UFFFD_ANTEIL:
        status = "kritisch: Font-Encoding"
    elif bilder > MAX_BILDER_BEI_WENIG_TEXT and woerter < MIN_WOERTER_BEI_BILDERN:
        status = "warnung: viele Bilder, wenig Text"
    else:
        status = "ok"
    return {"chars": chars, "woerter": woerter, "n_ufffd": n_ufffd,
            "ufffd_anteil": anteil, "bild_tags": bilder, "status": status}


if DATEN_QUELLE == "markdown":
    md_report = []
    for row in master_label_df[["paper_id", "year"]].itertuples(index=False):
        md_path = MARKDOWN_ROOT / row.year / f"{row.paper_id}.md"
        eintrag = {"paper_id": row.paper_id, "year": row.year}
        if not md_path.exists():
            md_report.append({**eintrag, "status": "fehlt"})
            continue
        try:
            md_report.append({**eintrag, **bewerte_markdown(md_path.read_text(encoding="utf-8"))})
        except Exception as e:
            md_report.append({**eintrag, "status": f"lesefehler: {e}"})

    md_report_df = pd.DataFrame(md_report, columns=[
        "paper_id", "year", "status", "chars", "woerter", "n_ufffd", "ufffd_anteil", "bild_tags"])
    md_report_df.to_csv(VALIDATION_ROOT / "markdown_report.csv", index=False)

    print(md_report_df["status"].value_counts().to_string())

    # Verteilung als Grundlage fuer die Wahl der Schwellen
    vorhanden = md_report_df.dropna(subset=["chars"])
    print("\nVerteilung U+FFFD-Anteil:")
    print(vorhanden["ufffd_anteil"].describe(percentiles=[0.5, 0.9, 0.99]).to_string())
    print(f"Dokumente mit mindestens einem U+FFFD: {(vorhanden['n_ufffd'] > 0).sum()}")
    print(f"Dokumente mit mindestens einem Bild-Tag: {(vorhanden['bild_tags'] > 0).sum()}")

    auffaellig = md_report_df[md_report_df["status"] != "ok"]
    if not auffaellig.empty:
        with pd.option_context("display.max_rows", None):
            display(auffaellig)
else:
    print(f"Uebersprungen (DATEN_QUELLE = '{DATEN_QUELLE}').")

## 3. Label-Qualitaet

Prueft die Rohlabels (`master_label_raw_df`, vor dem Mapping) gegen das Codebook
aus `Criteria_Definitions.docx` und auf logische Konsistenz zwischen Kriterien.

In [ ]:
# Zulaessige Werte laut Codebook; NaN steht fuer N/A
ERLAUBTE_WERTE = {
    "is_dataset_paper": {"Yes", "No"},
    "open_access": {"Yes", "No", "Unknown"},
    "code_required": {"Yes", "No", "Unknown"},
    "data_required": {"Yes", "No", "Unknown"},
    "hardware_required": {"Yes", "No", "Unknown", "N/A"},
    "software_required": {"Yes", "No", "Unknown", "N/A"},
    "code_available": {"Yes", "No", "Partially", "Upon_request", "Unknown", "N/A"},
    "data_available": {"Yes", "No", "Partially", "Upon_request", "Unknown", "N/A"},
}

raw = master_label_raw_df.copy()
befunde = []


def melde(pruefung: str, maske: pd.Series, detail: str = ""):
    for p_id in raw.loc[maske, "paper_id"]:
        befunde.append({"pruefung": pruefung, "paper_id": p_id, "detail": detail})


# A: Werte ausserhalb des Codebooks
for col, erlaubt in ERLAUBTE_WERTE.items():
    werte = raw[col].fillna("N/A").astype(str)
    melde(f"{col}: unzulaessiger Wert", ~werte.isin(erlaubt))

# B: paper_type - Formatierung und Kategorien
pt = raw["paper_type"].fillna("").astype(str)
melde("paper_type: leer", pt.str.strip() == "")
melde("paper_type: Leerzeichen am Rand", pt != pt.str.strip())
melde("paper_type: Leerzeichen am Semikolon", pt.str.contains(r"\s;|;\s", regex=True))
melde("paper_type: doppeltes Leerzeichen", pt.str.contains("  ", regex=False))
kategorien = pt.str.split(";").apply(lambda xs: {x.strip() for x in xs if x.strip()})
melde("paper_type: unbekannte Kategorie", kategorien.apply(lambda k: bool(k - set(PAPER_TYPE_CATEGORIES))))

# C: Logische Konsistenz zwischen Kriterien
for art in ["code", "data"]:
    req, avail, url = raw[f"{art}_required"], raw[f"{art}_available"], raw[f"{art}_url"]
    melde(f"{art}: nicht benoetigt, aber verfuegbar", req.isin(["No", "Unknown"]) & avail.eq("Yes"))
    melde(f"{art}: benoetigt, aber available = N/A", req.eq("Yes") & avail.isna())
    melde(f"{art}: verfuegbar, aber keine URL", avail.eq("Yes") & url.isna())
    melde(f"{art}: URL, aber nicht verfuegbar", avail.isin(["No", "Unknown"]) & url.notna(),
          "erwartbar bei leeren oder nur README enthaltenden Repositories")

befunde_df = pd.DataFrame(befunde, columns=["pruefung", "paper_id", "detail"])
befunde_df.to_csv(VALIDATION_ROOT / "label_report.csv", index=False)

if befunde_df.empty:
    print(f"Keine Befunde in {len(raw)} Zeilen.")
else:
    print(befunde_df["pruefung"].value_counts().to_string())

## 4. Determinismus der Embeddings

Prueft, ob Ollama fuer identischen Text bitgleiche Vektoren liefert.
Zwei zufaellig gezogene Studien (Seed aus dem Cockpit) werden zweimal direkt
hintereinander vektorisiert und zusaetzlich mit den gespeicherten Embeddings
aus Segment 3 verglichen. Der zweite Vergleich deckt ab, ob das Modell in
einer anderen Sitzung (anderer Ladezeitpunkt) dieselben Vektoren erzeugt.
Vektorisiert wird wie in Segment 3 ueber `embed_chunk` (inkl. Nachteilen);
gespeicherte Vektoren werden ueber den Chunk-Text zugeordnet.

Voraussetzung: laufender Ollama-Server, fuer Vergleich B die Embeddings aus Segment 3.

In [ ]:
N_DOKUMENTE = 2


def vergleiche(a: list, b: list) -> dict:
    """Bitgleichheit, groesste absolute Abweichung und Kosinus-Distanz (1 - cos) zweier Vektoren."""
    va, vb = np.asarray(a, dtype=np.float64), np.asarray(b, dtype=np.float64)
    if va.shape != vb.shape:
        return {"bitgleich": False, "max_abs_diff": np.nan, "cos_distanz": np.nan}
    if np.array_equal(va, vb):
        return {"bitgleich": True, "max_abs_diff": 0.0, "cos_distanz": 0.0}
    return {
        "bitgleich": False,
        "max_abs_diff": float(np.max(np.abs(va - vb))),
        "cos_distanz": float(1 - va @ vb / (np.linalg.norm(va) * np.linalg.norm(vb))),
    }


def vektorisiere(chunks: list, beschreibung: str) -> list:
    """Liste aller (Teil-)Chunks als {"text", "vector"}; None-Eintraege bei Fehlern."""
    teile = []
    for c in tqdm(chunks, desc=beschreibung):
        ergebnis = embed_chunk(c["text"])
        teile += ergebnis if ergebnis is not None else [{"text": c["text"], "vector": None}]
    return teile


if not check_ollama():
    print("Uebersprungen: Ollama-Server nicht erreichbar.")
else:
    quelle_dir, endung = (MARKDOWN_ROOT, ".md") if DATEN_QUELLE == "markdown" else (STUDIEN_ROOT, ".pdf")
    stichprobe = master_label_df[["paper_id", "year"]].sample(n=N_DOKUMENTE, random_state=GLOBAL_SEED)

    ergebnisse = []
    for row in stichprobe.itertuples(index=False):
        quelle = quelle_dir / row.year / f"{row.paper_id}{endung}"
        if not quelle.exists():
            print(f"{row.paper_id}: Quelldatei fehlt, uebersprungen")
            continue

        chunks, _ = extract_pdf_structure(quelle)
        lauf_1 = vektorisiere(chunks, f"{row.paper_id} Lauf 1")
        lauf_2 = vektorisiere(chunks, f"{row.paper_id} Lauf 2")

        # Gespeicherte Vektoren aus Segment 3 (andere Sitzung), zugeordnet ueber den Text
        gespeichert = {}
        emb_path = CURRENT_EMBEDDING_PATH / row.year / f"{row.paper_id}_embeddings.json"
        if emb_path.exists():
            daten = json.loads(emb_path.read_text(encoding="utf-8"))
            gespeichert = {c["text"]: c["vector"] for c in daten["chunks"]}

        if [t["text"] for t in lauf_1] != [t["text"] for t in lauf_2]:
            print(f"{row.paper_id}: Nachteilen lieferte in beiden Laeufen unterschiedliche Teile")
            continue

        for nr, (t1, t2) in enumerate(zip(lauf_1, lauf_2)):
            eintrag = {"paper_id": row.paper_id, "teil": nr, "chars": len(t1["text"])}
            if t1["vector"] is None or t2["vector"] is None:
                ergebnisse.append({**eintrag, "vergleich": "A: Lauf 1 vs. Lauf 2", "fehler": "kein Vektor"})
                continue
            ergebnisse.append({**eintrag, "vergleich": "A: Lauf 1 vs. Lauf 2", **vergleiche(t1["vector"], t2["vector"])})

            alt = gespeichert.get(t1["text"])
            if gespeichert and alt is None:
                ergebnisse.append({**eintrag, "vergleich": "B: gespeichert vs. Lauf 1",
                                   "fehler": "Text nicht in gespeicherten Chunks"})
            elif alt is not None:
                ergebnisse.append({**eintrag, "vergleich": "B: gespeichert vs. Lauf 1",
                                   **vergleiche(alt, t1["vector"])})

    det_df = pd.DataFrame(ergebnisse)
    det_df.to_csv(VALIDATION_ROOT / "embedding_determinismus.csv", index=False)

    print(f"\nStichprobe: {', '.join(stichprobe['paper_id'])}")
    if det_df.empty:
        print("Keine Vergleiche durchgefuehrt.")
    else:
        if "fehler" in det_df:
            fehler = det_df[det_df["fehler"].notna()]
            if not fehler.empty:
                print(fehler.groupby(["vergleich", "fehler"]).size().to_string())
            det_df = det_df[det_df["fehler"].isna()]
        det_df = det_df.astype({"bitgleich": bool})
        zusammenfassung = det_df.groupby("vergleich").agg(
            teile=("bitgleich", "size"),
            bitgleich=("bitgleich", "sum"),
            max_abs_diff=("max_abs_diff", "max"),
            max_cos_distanz=("cos_distanz", "max"),
        )
        print(zusammenfassung.to_string(float_format=lambda x: f"{x:.3e}"))
        if "B: gespeichert vs. Lauf 1" not in zusammenfassung.index:
            print("Vergleich B nicht moeglich: keine passenden gespeicherten Embeddings.")

## 5. Chunk-Groesse und Kontextfenster

Misst, wie viele Chunks bei verschiedenen Chunk-Groessen das Kontextfenster des
Embedding-Modells ueberschreiten. Zeichen und Tokens stehen in keinem festen
Verhaeltnis, daher wird direkt gegen Ollama gemessen. Vektoren werden nicht
gespeichert. Grundlage fuer die Wahl von `CHUNK_SIZE` im Cockpit.

Laufzeit: grob Anzahl Dokumente x Chunks je Dokument x Anzahl Groessen Ollama-Aufrufe.

In [ ]:
MESS_GROESSEN = [1200, 800, 600]   # Zeichen
N_MESS_DOKUMENTE = 10


def chunks_fuer_groesse(text: str, groesse: int) -> list:
    """Chunking wie im Cockpit, nur mit anderer Groesse (Overlap unveraendert)."""
    return RecursiveCharacterTextSplitter(
        chunk_size=groesse,
        chunk_overlap=min(CHUNK_OVERLAP, groesse // 2),
        separators=CHUNK_SEPARATORS,
    ).split_text(text)


if not check_ollama():
    print("Uebersprungen: Ollama-Server nicht erreichbar.")
elif DATEN_QUELLE != "markdown":
    print(f"Uebersprungen (DATEN_QUELLE = '{DATEN_QUELLE}').")
else:
    mess_stichprobe = master_label_df[["paper_id", "year"]].sample(n=N_MESS_DOKUMENTE, random_state=GLOBAL_SEED)
    texte = {
        row.paper_id: (MARKDOWN_ROOT / row.year / f"{row.paper_id}.md").read_text(encoding="utf-8")
        for row in mess_stichprobe.itertuples(index=False)
        if (MARKDOWN_ROOT / row.year / f"{row.paper_id}.md").exists()
    }
    n_aufrufe = sum(len(chunks_fuer_groesse(t, g)) for t in texte.values() for g in MESS_GROESSEN)
    print(f"Stichprobe: {len(texte)} Dokumente, {n_aufrufe} Ollama-Aufrufe")

    messung = []
    for groesse in MESS_GROESSEN:
        for p_id, text in tqdm(texte.items(), desc=f"{groesse} Zeichen"):
            for nr, chunk in enumerate(chunks_fuer_groesse(text, groesse)):
                try:
                    status = "ok" if get_embedding(chunk, raise_on_context=True) else "fehler"
                except KontextZuLangError:
                    status = "zu_lang"
                messung.append({"groesse": groesse, "paper_id": p_id, "chunk": nr,
                                "chars": len(chunk), "status": status,
                                "anfang": chunk[:120] if status == "zu_lang" else ""})

    mess_df = pd.DataFrame(messung)
    mess_df.to_csv(VALIDATION_ROOT / "chunk_kontextfenster.csv", index=False)

    zu_lang = mess_df["status"].eq("zu_lang")
    uebersicht = mess_df.assign(zu_lang=zu_lang).groupby("groesse").agg(
        chunks=("status", "size"),
        zu_lang=("zu_lang", "sum"),
        fehler=("status", lambda s: (s == "fehler").sum()),
    )
    uebersicht["anteil_zu_lang"] = uebersicht["zu_lang"] / uebersicht["chunks"]
    uebersicht["dokumente_betroffen"] = (
        mess_df[zu_lang].groupby("groesse")["paper_id"].nunique()
        .reindex(uebersicht.index, fill_value=0)
    )
    uebersicht["kuerzester_zu_langer"] = (
        mess_df[zu_lang].groupby("groesse")["chars"].min().reindex(uebersicht.index).astype("Int64")
    )
    print(uebersicht.sort_index(ascending=False).to_string(float_format=lambda x: f"{x:.4f}"))

    beispiele = mess_df[zu_lang].sort_values("chars").head(5)
    if not beispiele.empty:
        print("\nKuerzeste zu lange Chunks:")
        for b in beispiele.itertuples(index=False):
            print(f"  {b.groesse} | {b.paper_id} | {b.chars} Zeichen: {b.anfang!r}")

## 6. Entfernung des Literaturverzeichnisses

Prueft `entferne_literaturverzeichnis` aus dem Cockpit am gesamten Korpus:
ob die Ueberschrift gefunden wird, wie viel Text entfernt wird, ob Anhaenge
erhalten bleiben, und wie sich Links auf Haupttext und Literaturverzeichnis
verteilen. Letzteres prueft die Annahme, dass Repository- und Datensatz-Links
im Literaturverzeichnis ueberwiegend fremde Arbeiten betreffen.

Ein sehr hoher entfernter Anteil deutet auf eine zu frueh erkannte Ueberschrift hin.

In [2]:
MAX_ENTFERNT_ANTEIL = 0.5   # Darueber: Dokument manuell pruefen

LINK = re.compile(r"https?://\S+|github\.com/\S+", re.IGNORECASE)
REPO_LINK = re.compile(r"github\.com|gitlab\.com|bitbucket\.org|zenodo\.org|osf\.io|figshare\.com", re.IGNORECASE)


def lade_text(paper_id: str, jahr: str):
    """Volltext wie in extract_pdf_structure, aber vor dem Entfernen."""
    if DATEN_QUELLE == "markdown":
        pfad = MARKDOWN_ROOT / jahr / f"{paper_id}.md"
        return pfad.read_text(encoding="utf-8") if pfad.exists() else None
    pfad = STUDIEN_ROOT / jahr / f"{paper_id}.pdf"
    if not pfad.exists():
        return None
    with fitz.open(pfad) as doc:
        return "\n".join(page.get_text("text") for page in doc)


ref_report = []
for row in tqdm(master_label_df[["paper_id", "year"]].itertuples(index=False),
                total=len(master_label_df), desc="Literaturverzeichnis"):
    text = lade_text(row.paper_id, row.year)
    if text is None:
        ref_report.append({"paper_id": row.paper_id, "status": "quelle_fehlt"})
        continue
    rest, info = entferne_literaturverzeichnis(text, rohtext=(DATEN_QUELLE == "pdf_raw"))
    entfernt = ""
    if info["references_gefunden"]:
        entfernt = text[info["references_start"]:info["references_start"] + info["references_chars"]]
    ref_report.append({
        "paper_id": row.paper_id,
        "status": "ok" if info["references_gefunden"] else "keine_ueberschrift",
        **info,
        "entfernt_anteil": info["references_chars"] / len(text) if text else 0.0,
        "links_haupttext": len(LINK.findall(rest)),
        "links_references": len(LINK.findall(entfernt)),
        "repo_links_haupttext": len(REPO_LINK.findall(rest)),
        "repo_links_references": len(REPO_LINK.findall(entfernt)),
    })

ref_df = pd.DataFrame(ref_report)
ref_df.to_csv(VALIDATION_ROOT / f"references_report_{DATEN_QUELLE}.csv", index=False)

print(ref_df["status"].value_counts().to_string())
ok = ref_df[ref_df["status"] == "ok"]
print(f"\nAnhang nach dem Literaturverzeichnis erhalten: {int(ok['anhang_erhalten'].sum())} Dokumente")
print("Entfernter Anteil am Text:")
print(ok["entfernt_anteil"].describe(percentiles=[0.05, 0.5, 0.95]).to_string(float_format=lambda x: f"{x:.3f}"))

print("\nLinks gesamt (Haupttext / Literaturverzeichnis):")
print(f"  alle Links:       {int(ok['links_haupttext'].sum())} / {int(ok['links_references'].sum())}")
print(f"  Repository-Links: {int(ok['repo_links_haupttext'].sum())} / {int(ok['repo_links_references'].sum())}")

for titel, auswahl in [
    ("Ohne erkannte Ueberschrift", ref_df[ref_df["status"] == "keine_ueberschrift"]),
    (f"Entfernter Anteil ueber {MAX_ENTFERNT_ANTEIL:.0%}", ok[ok["entfernt_anteil"] > MAX_ENTFERNT_ANTEIL]),
]:
    if not auswahl.empty:
        print(f"\n{titel}: {len(auswahl)}")
        print(", ".join(auswahl["paper_id"].head(30)))

Literaturverzeichnis:   1%|▎                    | 9/654 [00:01<01:55,  5.57it/s]

MuPDF error: format error: object is not a stream

MuPDF error: syntax error: invalid ICC colorspace

MuPDF error: format error: object is not a stream

MuPDF error: syntax error: invalid ICC colorspace

MuPDF error: format error: object is not a stream

MuPDF error: syntax error: invalid ICC colorspace

MuPDF error: format error: object is not a stream

MuPDF error: syntax error: invalid ICC colorspace

MuPDF error: format error: object is not a stream

MuPDF error: syntax error: invalid ICC colorspace

MuPDF error: format error: object is not a stream

MuPDF error: syntax error: invalid ICC colorspace

MuPDF error: format error: object is not a stream

MuPDF error: syntax error: invalid ICC colorspace

MuPDF error: format error: object is not a stream

MuPDF error: syntax error: invalid ICC colorspace

MuPDF error: format error: object is not a stream

MuPDF error: syntax error: invalid ICC colorspace

MuPDF error: format error: object is not a stream

MuPDF error: syntax error: inva

Literaturverzeichnis:  14%|██▊                 | 93/654 [00:14<01:16,  7.38it/s]

MuPDF error: format error: object is not a stream



Literaturverzeichnis:  29%|█████▍             | 188/654 [00:27<00:43, 10.66it/s]

...

MuPDF error: syntax error: image width is zero (or less)



Literaturverzeichnis:  59%|███████████▏       | 386/654 [01:00<00:38,  6.95it/s]

MuPDF error: syntax error: image width is zero (or less)



Literaturverzeichnis: 100%|███████████████████| 654/654 [01:45<00:00,  6.20it/s]

status
ok    654

Anhang nach dem Literaturverzeichnis erhalten: 267 Dokumente
Entfernter Anteil am Text:
count   654.000
mean      0.180
std       0.052
min       0.063
5%        0.112
50%       0.173
95%       0.272
max       0.469

Links gesamt (Haupttext / Literaturverzeichnis):
  alle Links:       2377 / 13622
  Repository-Links: 284 / 179


In [3]:
# Plausibilitaetspruefung der Schnittgrenzen (nur Markdown, setzt Abschnitt 6 voraus).
# A: Welche Zeilen wurden als Anhang-Beginn erkannt?
# B: Enthaelt der entfernte Teil weitere Ueberschriften, die wie Abschnitte aussehen?
#    Das waeren moegliche, nicht erkannte Anhaenge. Seitenkoepfe ("55:24 • Autor et al.")
#    und von pymupdf4llm als Ueberschrift gesetzte Literaturfragmente ("4, 2 (2016), 41.")
#    werden ignoriert.
# C: Dokumente mit dem hoechsten entfernten Anteil, mit Typ und Zahl der Literatureintraege.
UEBERSCHRIFT = re.compile(r"^[ \t]*#+[ \t]*(.+?)[ \t]*$", re.MULTILINE)
SEITENKOPF = re.compile(r"•|^\d+:\d+")
ABSCHNITT = re.compile(r"^(?:\*\*|_)?[ \t]*(?:(?:[A-Z]|\d+)(?:\.\d+)*\.?[ \t]+[A-Za-z]|[^\n]*\b(?i:appendix|appendices|supplementary)\b)")
EINTRAG = re.compile(r"^[ \t]*(?:-[ \t]*)?\[\d+\]", re.MULTILINE)

jahr_von = master_label_df.set_index("paper_id")["year"]
anhang_koepfe, verdacht, eintraege = [], [], {}

for row in ref_df[ref_df["status"] == "ok"].itertuples(index=False):
    text = lade_text(row.paper_id, jahr_von[row.paper_id])
    start, laenge = int(row.references_start), int(row.references_chars)
    eintraege[row.paper_id] = len(EINTRAG.findall(text[start:start + laenge]))
    if row.anhang_erhalten:
        anhang_koepfe.append((row.paper_id, text[start + laenge:].splitlines()[0].strip()))
    koepfe = [k for k in UEBERSCHRIFT.findall(text[start:start + laenge])[1:]
              if not SEITENKOPF.search(k) and ABSCHNITT.search(k)]
    if koepfe:
        verdacht.append({"paper_id": row.paper_id, "n_ueberschriften": len(koepfe),
                         "repo_links_references": row.repo_links_references,
                         "beispiele": " | ".join(koepfe[:3])})

anhang_df = pd.DataFrame(anhang_koepfe, columns=["paper_id", "anhang_ueberschrift"])
verdacht_df = pd.DataFrame(verdacht, columns=["paper_id", "n_ueberschriften", "repo_links_references", "beispiele"])
anhang_df.to_csv(VALIDATION_ROOT / f"references_anhaenge_{DATEN_QUELLE}.csv", index=False)
verdacht_df.to_csv(VALIDATION_ROOT / f"references_verdacht_{DATEN_QUELLE}.csv", index=False)

print(f"A: Erkannte Anhang-Ueberschriften: {len(anhang_df)} (Stichprobe)")
for r in anhang_df.sample(n=min(15, len(anhang_df)), random_state=GLOBAL_SEED).itertuples(index=False):
    print(f"  {r.paper_id}: {r.anhang_ueberschrift[:90]!r}")

print(f"\nB: Entfernte Teile mit weiteren Ueberschriften: {len(verdacht_df)}")
with pd.option_context("display.max_colwidth", 100, "display.width", 200):
    print(verdacht_df.sort_values("n_ueberschriften", ascending=False).head(20).to_string(index=False))

anteil_df = ref_df[ref_df["status"] == "ok"].nlargest(10, "entfernt_anteil").copy()
anteil_df["eintraege"] = anteil_df["paper_id"].map(eintraege)
anteil_df["paper_type"] = anteil_df["paper_id"].map(master_label_df.set_index("paper_id")["paper_type"])
print(f"\nC: Hoechster entfernter Anteil (Median Literatureintraege im Korpus: {pd.Series(eintraege).median():.0f})")
with pd.option_context("display.max_colwidth", 60, "display.width", 200):
    print(anteil_df[["paper_id", "entfernt_anteil", "references_chars", "eintraege", "paper_type"]]
          .to_string(index=False, float_format=lambda x: f"{x:.3f}"))


MuPDF error: format error: object is not a stream

MuPDF error: syntax error: invalid ICC colorspace

MuPDF error: format error: object is not a stream

MuPDF error: syntax error: invalid ICC colorspace

MuPDF error: format error: object is not a stream

MuPDF error: syntax error: invalid ICC colorspace

MuPDF error: format error: object is not a stream

MuPDF error: syntax error: invalid ICC colorspace

MuPDF error: format error: object is not a stream

MuPDF error: syntax error: invalid ICC colorspace

MuPDF error: format error: object is not a stream

MuPDF error: syntax error: invalid ICC colorspace

MuPDF error: format error: object is not a stream

MuPDF error: syntax error: invalid ICC colorspace

MuPDF error: format error: object is not a stream

MuPDF error: syntax error: invalid ICC colorspace

MuPDF error: format error: object is not a stream

MuPDF error: syntax error: invalid ICC colorspace

MuPDF error: format error: object is not a stream

MuPDF error: syntax error: inva

In [4]:
# Abgleich Markdown gegen PDF-Rohtext: Anhaenge, die im Markdown erkannt werden, im
# Rohtext aber nicht. Zeigt die Rohtext-Zeilen an der Stelle der Markdown-Ueberschrift,
# damit die Rohtext-Regel gezielt angepasst werden kann. Unabhaengig von DATEN_QUELLE.
N_BEISPIELE = 8


def rohtext(paper_id: str, jahr: str) -> str:
    with fitz.open(STUDIEN_ROOT / jahr / f"{paper_id}.pdf") as doc:
        return "\n".join(page.get_text("text") for page in doc)


fitz.TOOLS.mupdf_display_errors(False)
abweichend = []
for row in tqdm(master_label_df[["paper_id", "year"]].itertuples(index=False),
                total=len(master_label_df), desc="Markdown vs. Rohtext"):
    md_pfad = MARKDOWN_ROOT / row.year / f"{row.paper_id}.md"
    if not md_pfad.exists():
        continue
    md = md_pfad.read_text(encoding="utf-8")
    _, md_info = entferne_literaturverzeichnis(md)
    if not md_info["anhang_erhalten"]:
        continue
    roh = rohtext(row.paper_id, row.year)
    _, roh_info = entferne_literaturverzeichnis(roh, rohtext=True)
    if not roh_info["anhang_erhalten"]:
        kopf = md[md_info["references_start"] + md_info["references_chars"]:].splitlines()[0]
        abweichend.append((row.paper_id, kopf, roh))

print(f"Anhang im Markdown erkannt, im Rohtext nicht: {len(abweichend)}")
for p_id, kopf, roh in abweichend[:N_BEISPIELE]:
    titel = re.sub(r"[#*_]", "", kopf).strip()
    wort = max(titel.split(), key=len) if titel.split() else titel
    pos = roh.lower().rfind(wort.lower())
    print(f"\n{p_id} | Markdown: {kopf.strip()[:80]!r}")
    if pos < 0:
        print(f"  Suchwort {wort!r} im Rohtext nicht gefunden")
        continue
    zeilen_start = roh.rfind("\n", 0, max(0, pos - 150)) + 1
    for zeile in roh[zeilen_start:pos + 200].splitlines()[:8]:
        print(f"  | {zeile!r}")

Markdown vs. Rohtext: 100%|███████████████████| 654/654 [00:43<00:00, 15.16it/s]

Anhang im Markdown erkannt, im Rohtext nicht: 1

2025_Dec_172 | Markdown: '## A SEARCH DETAILS CORRESPONDING TO EACH SOURCE'
  | 'Proc. ACM Interact. Mob. Wearable Ubiquitous Technol., Vol. 9, No. 4, Article 172. Publication date: December 2025.'
  | ''
  | '172:40'
  | '•'
  | 'Fang et al.'
  | 'A'
  | 'SEARCH DETAILS CORRESPONDING TO EACH SOURCE'
  | 'ACM Digital Library'


## 7. Bestandsaufnahme der Embedding- und Fusionsordner

Listet alle vorhandenen Ordner unter `daten/embeddings` und `daten/fusionsdaten` mit
Inhalt und Herkunft. Der Ordnername ergibt sich aus Datenquelle, Modell und
`STRATEGIE_NAME` (Chunk-Groesse, Overlap, `_prefixed` bei Dokument-Praefix,
`_norefs` bei entferntem Literaturverzeichnis). Ordner ohne Manifest stammen aus
Versionen vor dem Embedding-Manifest; leere Ordner hat fruehere Versionen des
Cockpits beim Start angelegt.

In [ ]:
def embedding_ordner_info(ordner: Path) -> dict:
    dateien = sorted(ordner.rglob("*_embeddings.json"))
    info = {
        "ordner": str(ordner.relative_to(EMBEDDING_BASE_ROOT)),
        "aktuell": ordner == CURRENT_EMBEDDING_PATH,
        "json_dateien": len(dateien),
        "manifest": (ordner / "embedding_manifest.csv").exists(),
        "lauf_abgebrochen": (ordner / "embedding_lauf.json").exists(),
        "erstellt_von": None, "erstellt_bis": None, "konfiguration": None,
    }
    if dateien:
        zeiten = [d.stat().st_mtime for d in dateien]
        info["erstellt_von"] = datetime.fromtimestamp(min(zeiten)).strftime("%Y-%m-%d %H:%M")
        info["erstellt_bis"] = datetime.fromtimestamp(max(zeiten)).strftime("%Y-%m-%d %H:%M")
        meta = json.loads(dateien[0].read_text(encoding="utf-8")).get("metadata", {})
        cfg = meta.get("embed_config")
        info["konfiguration"] = (
            f"prefix={cfg.get('prefix')!r}, refs_entfernt={cfg.get('references_entfernen', False)}, "
            f"nachteilen={'nachteilen' in cfg}" if cfg else "alt (ohne gespeicherte Konfiguration)"
        )
    if info["manifest"]:
        m = pd.read_csv(ordner / "embedding_manifest.csv")
        info["manifest_studien"] = len(m)
        info["konfigurationen_im_manifest"] = m["config_sha256"].nunique()
    return info


# Ordnerebene: <quelle>/<modell>/<strategie>
emb_ordner = [p for p in EMBEDDING_BASE_ROOT.glob("*/*/*") if p.is_dir()]
emb_df = pd.DataFrame([embedding_ordner_info(p) for p in sorted(emb_ordner)])
with pd.option_context("display.max_colwidth", 80, "display.width", 250):
    print(emb_df.to_string(index=False))

fus = []
for ordner in sorted(p for p in FUSION_BASE_ROOT.glob("*/*/*/*") if p.is_dir()):
    x_pfad = ordner / "X_data.npy"
    fus.append({
        "ordner": str(ordner.relative_to(FUSION_BASE_ROOT)),
        "X_vorhanden": x_pfad.exists(),
        "manifest": (ordner / "fusion_manifest.json").exists(),
        "erstellt": datetime.fromtimestamp(x_pfad.stat().st_mtime).strftime("%Y-%m-%d %H:%M") if x_pfad.exists() else None,
    })
print()
with pd.option_context("display.max_colwidth", 90, "display.width", 250):
    print(pd.DataFrame(fus).to_string(index=False))